# Lab 04 — Controlled Schema Evolution

Notebook 08 proved that Delta rejects unapproved schema drift. This notebook demonstrates the governed alternative: approve contract v2, evolve the Delta table intentionally, validate every change, and preserve idempotent reruns.

## Learning objectives

- add an approved nullable column with per-write `mergeSchema`;
- add approved columns with explicit per-write `mergeSchema`, which is supported by Databricks serverless compute;
- widen `INT` to `BIGINT` with Delta type widening enabled;
- compare contract v1 and contract v2;
- prove that existing rows remain readable and receive `NULL` for new fields;
- explain why session-wide `autoMerge` is intentionally avoided in this notebook;
- replay an evolved batch without inserting duplicates.

> This notebook uses a dedicated demonstration table. It does not alter the production Silver table.

## 1. Load shared configuration

The configuration notebook supplies the catalog, schema, volume, table names, and validation settings. The curated Silver transaction table created by notebook 04 is the trusted source for this controlled experiment.

In [0]:
%run ./lab04_00_config

# Lab 04 — Configuration and Unity Catalog Setup

This notebook:
- defines Lab 4 parameters;
- creates the Unity Catalog catalog and schema when permitted;
- creates a managed or external volume;
- builds source, staging, landing, schema, checkpoint, quarantine, and test folders;
- defines all Bronze, Silver, SCD, and demonstration table names.

Run this notebook first. It is a setup notebook and does not need to be scheduled in the production Job.

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Catalog ready: dbr_dev
Schema ready: dbr_dev.parvinbadalov
Volume ready: dbr_dev.parvinbadalov.lab04_silver_quality (external)
External location: abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/lab04_silver_quality


Created or verified 13 Lab 4 folders under /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
  source: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source
  staging_initial: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial
  staging_incremental: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/incremental
  staging_evolved: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/evolved
  staging_invalid: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/invalid
  landing: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing
  schema_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/schema/bronze
  checkpoint_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/bronze
  checkpoint_silver: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/silver
  quarantine: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine
  schema_mismatch: /Volumes/dbr_dev/parvinba

Lab 4 table names configured:
  bronze: dbr_dev.parvinbadalov.lab04_bronze_retail
  silver_transactions: dbr_dev.parvinbadalov.lab04_silver_transactions
  quarantine: dbr_dev.parvinbadalov.lab04_quarantine
  quality_metrics: dbr_dev.parvinbadalov.lab04_quality_metrics
  product_scd0: dbr_dev.parvinbadalov.lab04_product_scd0
  product_scd1: dbr_dev.parvinbadalov.lab04_product_scd1
  product_scd2: dbr_dev.parvinbadalov.lab04_product_scd2
  product_scd3: dbr_dev.parvinbadalov.lab04_product_scd3
  product_scd4_current: dbr_dev.parvinbadalov.lab04_product_current
  product_scd4_history: dbr_dev.parvinbadalov.lab04_product_history
  product_scd6: dbr_dev.parvinbadalov.lab04_product_scd6
  schema_demo: dbr_dev.parvinbadalov.lab04_schema_demo
  column_mapping_demo: dbr_dev.parvinbadalov.lab04_column_mapping_demo


Upload the workbook to: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Expected columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Active contract: v1
Schema policy: fail
Trigger: availableNow; maximum files per trigger: 50


Volume validation succeeded; 6 top-level entries found.
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/


Active data contract: online_retail v1


contract,version,status,column_count
v1,1,active,8
v2,2,proposed,10


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

evolution_table = f"{catalog}.{schema}.lab04_schema_evolution_demo"
silver_table = table_names["silver_transactions"]

if "contract_v1" not in globals() or "contract_v2" not in globals():
    raise NameError(
        "contract_v1/contract_v2 are not loaded. Update lab04_00_config so it "
        "loads contracts/online_retail_v1.yml and online_retail_v2.yml."
    )

baseline_contract = contract_v1
evolved_contract = contract_v2
baseline_contract_version = f"v{baseline_contract['contract']['version']}"
evolved_contract_version = f"v{evolved_contract['contract']['version']}"

if baseline_contract_version != "v1":
    raise AssertionError(
        f"Schema-evolution baseline must be v1; loaded {baseline_contract_version}."
    )
if evolved_contract_version != "v2":
    raise AssertionError(
        f"Schema-evolution target must be v2; loaded {evolved_contract_version}."
    )
if int(evolved_contract["contract"].get("supersedes", -1)) != 1:
    raise AssertionError("Contract v2 must declare supersedes: 1.")

v1_contract_columns = {
    item["name"]: item
    for item in baseline_contract["schema"]["columns"]
}
v2_contract_columns = {
    item["name"]: item
    for item in evolved_contract["schema"]["columns"]
}

loyalty_values = v2_contract_columns["loyalty_tier"].get("allowed_values", [])
sales_channel_values = v2_contract_columns["sales_channel"].get("allowed_values", [])

if not loyalty_values:
    raise AssertionError("Contract v2 must define allowed_values for loyalty_tier.")
if not sales_channel_values:
    raise AssertionError("Contract v2 must define allowed_values for sales_channel.")

print(f"Silver source: {silver_table}")
print(f"Evolution target: {evolution_table}")
print(f"Runtime widget contract: {contract_version}")
print(
    f"Evolution demonstration: {baseline_contract_version} "
    f"({baseline_contract['contract']['status']}) → "
    f"{evolved_contract_version} ({evolved_contract['contract']['status']})"
)
print(f"Allowed loyalty tiers from YAML: {loyalty_values}")
print(f"Allowed sales channels from YAML: {sales_channel_values}")


Silver source: dbr_dev.parvinbadalov.lab04_silver_transactions
Evolution target: dbr_dev.parvinbadalov.lab04_schema_evolution_demo
Runtime widget contract: v1
Evolution demonstration: v1 (active) → v2 (proposed)
Allowed loyalty tiers from YAML: ['STANDARD', 'SILVER', 'GOLD']
Allowed sales channels from YAML: ['ONLINE', 'MARKETPLACE', 'DIRECT']


## 2. Validate the Silver prerequisite and contract files

This notebook always demonstrates the **v1 → v2 contract transition**. The global `contract_version` widget is still printed for observability, but it does not collapse this two-version experiment into a single version.

The repository YAML files are the governance source of truth for:
- the v1 and v2 version numbers;
- the `supersedes` relationship;
- fields added by v2;
- the `Quantity` type widening;
- allowed values for `loyalty_tier` and `sales_channel`.


In [0]:
if not spark.catalog.tableExists(silver_table):
    raise FileNotFoundError(
        f"Silver table {silver_table} does not exist. "
        "Run lab04_04_silver_merge.ipynb first."
    )

required_columns = {
    "transaction_line_id", "invoice_no", "stock_code", "quantity",
    "unit_price", "invoice_timestamp", "country", "source_batch_id",
}
silver_source_df = spark.table(silver_table)
missing_columns = sorted(required_columns - set(silver_source_df.columns))
if missing_columns:
    raise AssertionError(f"Silver prerequisite is missing columns: {missing_columns}")

silver_count = silver_source_df.count()
if silver_count < 16:
    raise AssertionError(f"At least 16 Silver rows are required; found {silver_count}.")

print(f"✅ Silver prerequisite ready: {silver_count:,} rows.")

✅ Silver prerequisite ready: 315,101 rows.


## 3. Create a deterministic contract-v1 table

The target begins with `quantity INT` and without the two v2 fields. Type widening is enabled at the table level, but no schema change occurs until an approved write explicitly requests it. The table is recreated on each run so the demonstration is repeatable.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {evolution_table}")
spark.sql(
    f"""
    CREATE TABLE {evolution_table} (
        transaction_line_id STRING NOT NULL,
        invoice_no STRING NOT NULL,
        stock_code STRING NOT NULL,
        quantity INT NOT NULL,
        unit_price DECIMAL(18, 4) NOT NULL,
        invoice_timestamp TIMESTAMP NOT NULL,
        country STRING NOT NULL,
        source_batch_id STRING NOT NULL,
        contract_version STRING NOT NULL,
        evolved_at TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableTypeWidening' = 'true',
        'lab.contract.version' = '{baseline_contract_version}',
        'lab.contract.source' = 'contracts/online_retail_v1.yml',
        'lab.evolution.policy' = 'approved_changes_only'
    )
    """
)

rank_window = Window.orderBy("transaction_line_id")
ranked_source_df = silver_source_df.withColumn(
    "_sample_row_number",
    F.row_number().over(rank_window),
)

def sample_range(start_row, end_row):
    return ranked_source_df.filter(
        F.col("_sample_row_number").between(start_row, end_row)
    )

def project_contract_batch(
    dataframe,
    quantity_type,
    contract,
    loyalty=None,
    channel=None,
):
    projected = dataframe.select(
        F.col("transaction_line_id").cast("string"),
        F.col("invoice_no").cast("string"),
        F.col("stock_code").cast("string"),
        F.col("quantity").cast(quantity_type).alias("quantity"),
        F.col("unit_price").cast("decimal(18,4)"),
        F.col("invoice_timestamp").cast("timestamp"),
        F.col("country").cast("string"),
        F.col("source_batch_id").cast("string"),
        F.lit(contract).alias("contract_version"),
        F.current_timestamp().alias("evolved_at"),
    )
    if loyalty is not None:
        projected = projected.withColumn(
            "loyalty_tier",
            F.lit(loyalty).cast("string"),
        )
    if channel is not None:
        projected = projected.withColumn(
            "sales_channel",
            F.lit(channel).cast("string"),
        )
    return projected

v1_batch_df = project_contract_batch(
    sample_range(1, 4),
    "int",
    baseline_contract_version,
)
v1_batch_df.write.format("delta").mode("append").saveAsTable(evolution_table)

baseline_count = spark.table(evolution_table).count()
if baseline_count != 4:
    raise AssertionError(f"Expected 4 v1 rows, found {baseline_count}.")

print(f"✅ Contract-v1 baseline created with {baseline_count} rows.")
spark.table(evolution_table).printSchema()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Contract-v1 baseline created with 4 rows.
root
 |-- transaction_line_id: string (nullable = false)
 |-- invoice_no: string (nullable = false)
 |-- stock_code: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: decimal(18,4) (nullable = false)
 |-- invoice_timestamp: timestamp (nullable = false)
 |-- country: string (nullable = false)
 |-- source_batch_id: string (nullable = false)
 |-- contract_version: string (nullable = false)
 |-- evolved_at: timestamp (nullable = false)



## 4. Add `loyalty_tier` with `mergeSchema`

Contract v2 approves a nullable `loyalty_tier`. `mergeSchema` is applied only to this write, making the approval visible in code and limiting the change to one operation. Existing v1 rows remain valid and receive `NULL` for the new column.

In [0]:
loyalty_batch_df = project_contract_batch(
    sample_range(5, 8),
    "int",
    evolved_contract_version,
    loyalty=loyalty_values[0],
)

(
    loyalty_batch_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(evolution_table)
)

after_loyalty_df = spark.table(evolution_table)
if "loyalty_tier" not in after_loyalty_df.columns:
    raise AssertionError("loyalty_tier did not flow through mergeSchema.")
if after_loyalty_df.count() != 8:
    raise AssertionError("Unexpected count after loyalty_tier evolution.")

legacy_null_count = after_loyalty_df.filter(
    (F.col("contract_version") == baseline_contract_version)
    & F.col("loyalty_tier").isNull()
).count()
if legacy_null_count != baseline_count:
    raise AssertionError(
        "Existing v1 rows were not preserved with NULL loyalty_tier."
    )

invalid_loyalty_count = after_loyalty_df.filter(
    F.col("loyalty_tier").isNotNull()
    & ~F.col("loyalty_tier").isin(loyalty_values)
).count()
if invalid_loyalty_count:
    raise AssertionError(
        f"Found {invalid_loyalty_count} loyalty_tier values outside contract v2."
    )

print(
    f"✅ mergeSchema added loyalty_tier using YAML-approved value "
    f"{loyalty_values[0]!r}; existing rows were preserved."
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ mergeSchema added loyalty_tier using YAML-approved value 'STANDARD'; existing rows were preserved.


## 5. Add `sales_channel` with explicit `mergeSchema`

Contract v2 also approves the nullable `sales_channel` field. This notebook deliberately uses `.option("mergeSchema", "true")` on the individual write instead of changing the session-wide `spark.databricks.delta.schema.autoMerge.enabled` setting. The explicit option works on Databricks serverless compute, limits permission to one reviewed operation, and makes the evolution visible in code review.

In [0]:
channel_batch_df = project_contract_batch(
    sample_range(9, 12),
    "int",
    evolved_contract_version,
    loyalty=loyalty_values[min(1, len(loyalty_values) - 1)],
    channel=sales_channel_values[0],
)

(
    channel_batch_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(evolution_table)
)

channel_evolution_method = "per-write mergeSchema (serverless)"

after_channel_df = spark.table(evolution_table)
if "sales_channel" not in after_channel_df.columns:
    raise AssertionError(
        "sales_channel did not flow through controlled schema evolution."
    )
if after_channel_df.count() != 12:
    raise AssertionError("Unexpected count after sales_channel evolution.")

invalid_channel_count = after_channel_df.filter(
    F.col("sales_channel").isNotNull()
    & ~F.col("sales_channel").isin(sales_channel_values)
).count()
if invalid_channel_count:
    raise AssertionError(
        f"Found {invalid_channel_count} sales_channel values outside contract v2."
    )

print(
    f"✅ sales_channel added with {channel_evolution_method} using "
    f"YAML-approved value {sales_channel_values[0]!r}."
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ sales_channel added with per-write mergeSchema (serverless) using YAML-approved value 'ONLINE'.


## 6. Widen `quantity` from `INT` to `BIGINT`

Contract v2 approves an integer widening. With `delta.enableTypeWidening=true`, a `BIGINT` source can evolve an `INT` target when `mergeSchema` is requested. Widening is generally safe because every prior integer value fits in the wider type; narrowing would risk data loss and is not approved.

In [0]:
widen_batch_df = project_contract_batch(
    sample_range(13, 16),
    "long",
    evolved_contract_version,
    loyalty=loyalty_values[-1],
    channel=sales_channel_values[-1],
)

(
    widen_batch_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(evolution_table)
)

evolved_df = spark.table(evolution_table)
quantity_type = next(
    field.dataType.simpleString()
    for field in evolved_df.schema.fields
    if field.name == "quantity"
)

yaml_quantity_v1 = v1_contract_columns["Quantity"]["type"]
yaml_quantity_v2 = v2_contract_columns["Quantity"]["type"]

if (yaml_quantity_v1, yaml_quantity_v2) != ("integer", "long"):
    raise AssertionError(
        "Expected YAML contract evolution Quantity: integer → long; "
        f"found {yaml_quantity_v1} → {yaml_quantity_v2}."
    )
if quantity_type != "bigint":
    raise AssertionError(
        f"Expected quantity BIGINT after widening; found {quantity_type}."
    )
if evolved_df.count() != 16:
    raise AssertionError("Unexpected count after type widening.")

print(
    f"✅ Type widening completed from YAML contract "
    f"{yaml_quantity_v1} → {yaml_quantity_v2}; "
    f"Delta quantity is now {quantity_type}."
)
evolved_df.printSchema()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Type widening completed from YAML contract integer → long; Delta quantity is now bigint.
root
 |-- transaction_line_id: string (nullable = false)
 |-- invoice_no: string (nullable = false)
 |-- stock_code: string (nullable = false)
 |-- quantity: long (nullable = false)
 |-- unit_price: decimal(18,4) (nullable = false)
 |-- invoice_timestamp: timestamp (nullable = false)
 |-- country: string (nullable = false)
 |-- source_batch_id: string (nullable = false)
 |-- contract_version: string (nullable = false)
 |-- evolved_at: timestamp (nullable = false)
 |-- loyalty_tier: string (nullable = true)
 |-- sales_channel: string (nullable = true)



## 7. Compare contract v1 and v2 from YAML

The change list below is **not hardcoded**. It is derived directly from:

- `contracts/online_retail_v1.yml`
- `contracts/online_retail_v2.yml`

This proves that the repository contract files are executable governance artifacts rather than documentation-only files.


In [0]:
def contract_schema_map(contract):
    return {
        item["name"]: item
        for item in contract["schema"]["columns"]
    }

v1_schema_map = contract_schema_map(baseline_contract)
v2_schema_map = contract_schema_map(evolved_contract)

contract_changes = []
for field_name in sorted(set(v1_schema_map) | set(v2_schema_map)):
    v1_spec = v1_schema_map.get(field_name)
    v2_spec = v2_schema_map.get(field_name)

    if v1_spec is None:
        change_type = (
            "ADD_NULLABLE_COLUMN"
            if v2_spec.get("nullable", True)
            else "ADD_REQUIRED_COLUMN"
        )
        backward_compatible = bool(v2_spec.get("nullable", True))
        contract_changes.append(
            (
                field_name,
                None,
                v2_spec["type"],
                change_type,
                backward_compatible,
            )
        )

    elif v2_spec is None:
        contract_changes.append(
            (
                field_name,
                v1_spec["type"],
                None,
                "DROP_COLUMN",
                False,
            )
        )

    elif v1_spec["type"] != v2_spec["type"]:
        is_safe_widen = (
            v1_spec["type"] == "integer"
            and v2_spec["type"] == "long"
        )
        contract_changes.append(
            (
                field_name,
                v1_spec["type"],
                v2_spec["type"],
                "WIDEN_NUMERIC_TYPE" if is_safe_widen else "TYPE_CHANGE",
                is_safe_widen,
            )
        )

contract_changes_df = spark.createDataFrame(
    contract_changes,
    [
        "field_name",
        "v1_type",
        "v2_type",
        "change_type",
        "backward_compatible",
    ],
)

expected_change_names = {"Quantity", "loyalty_tier", "sales_channel"}
actual_change_names = {
    row["field_name"]
    for row in contract_changes_df.select("field_name").collect()
}
if actual_change_names != expected_change_names:
    raise AssertionError(
        f"Unexpected YAML contract diff. "
        f"Expected {sorted(expected_change_names)}, "
        f"found {sorted(actual_change_names)}."
    )

display(contract_changes_df.orderBy("field_name"))
print("✅ Contract v1 → v2 diff was derived directly from the YAML files.")


field_name,v1_type,v2_type,change_type,backward_compatible
Quantity,integer,long,WIDEN_NUMERIC_TYPE,true
loyalty_tier,null,string,ADD_NULLABLE_COLUMN,true
sales_channel,null,string,ADD_NULLABLE_COLUMN,true


✅ Contract v1 → v2 diff was derived directly from the YAML files.


## 8. Prove replay idempotency

Schema evolution and idempotency solve different problems. Evolution changes the permitted structure; `MERGE` prevents a retry of the same evolved records from inserting duplicates. The batch already exists, so a key-based replay must leave the target count unchanged.

In [0]:
count_before_replay = spark.table(evolution_table).count()

(
    DeltaTable.forName(spark, evolution_table)
    .alias("target")
    .merge(
        widen_batch_df.alias("source"),
        "target.transaction_line_id = source.transaction_line_id",
    )
    .whenNotMatchedInsertAll()
    .execute()
)

count_after_replay = spark.table(evolution_table).count()
if count_after_replay != count_before_replay:
    raise AssertionError(
        f"Replay changed count from {count_before_replay} to {count_after_replay}."
    )

print(f"✅ Idempotent replay preserved {count_after_replay} rows.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Idempotent replay preserved 16 rows.


## 9. Final validation and Delta history

The final checks prove that both approved columns exist, the numeric type widened, all 16 unique records remain, and the replay added nothing. Delta history provides auditable evidence of the writes that changed the schema.

In [0]:
final_df = spark.table(evolution_table)
final_columns = set(final_df.columns)
final_count = final_df.count()
distinct_key_count = (
    final_df.select("transaction_line_id").distinct().count()
)

invalid_loyalty_count = final_df.filter(
    F.col("loyalty_tier").isNotNull()
    & ~F.col("loyalty_tier").isin(loyalty_values)
).count()
invalid_channel_count = final_df.filter(
    F.col("sales_channel").isNotNull()
    & ~F.col("sales_channel").isin(sales_channel_values)
).count()

validation_rows = [
    (
        "v2_supersedes_v1",
        int(evolved_contract["contract"].get("supersedes", -1)) == 1,
        str(evolved_contract["contract"].get("supersedes")),
    ),
    (
        "yaml_contract_diff_matches_demo",
        actual_change_names == {"Quantity", "loyalty_tier", "sales_channel"},
        ", ".join(sorted(actual_change_names)),
    ),
    (
        "loyalty_tier_added",
        "loyalty_tier" in final_columns,
        "mergeSchema",
    ),
    (
        "sales_channel_added",
        "sales_channel" in final_columns,
        channel_evolution_method,
    ),
    (
        "quantity_widened",
        quantity_type == "bigint",
        quantity_type,
    ),
    (
        "loyalty_values_respect_v2_contract",
        invalid_loyalty_count == 0,
        str(invalid_loyalty_count),
    ),
    (
        "sales_channel_values_respect_v2_contract",
        invalid_channel_count == 0,
        str(invalid_channel_count),
    ),
    (
        "expected_row_count",
        final_count == 16,
        str(final_count),
    ),
    (
        "unique_business_keys",
        distinct_key_count == final_count,
        str(distinct_key_count),
    ),
    (
        "replay_idempotent",
        count_before_replay == count_after_replay,
        str(count_after_replay),
    ),
    (
        "compute_compatible_evolution",
        True,
        channel_evolution_method,
    ),
]
validation_df = spark.createDataFrame(
    validation_rows,
    ["validation", "passed", "evidence"],
)
display(validation_df.orderBy("validation"))

failed_validations = validation_df.filter(~F.col("passed")).count()
if failed_validations:
    raise AssertionError(
        f"Schema evolution validation failed: "
        f"{failed_validations} checks."
    )

display(
    final_df
    .select(
        "transaction_line_id",
        "quantity",
        "loyalty_tier",
        "sales_channel",
        "contract_version",
    )
    .orderBy("transaction_line_id")
)
display(spark.sql(f"DESCRIBE HISTORY {evolution_table}"))

print(
    f"✅ Controlled schema evolution validation passed using repository "
    f"contracts {baseline_contract_version} → {evolved_contract_version}."
)


validation,passed,evidence
compute_compatible_evolution,true,per-write mergeSchema (serverless)
expected_row_count,true,16
loyalty_tier_added,true,mergeSchema
loyalty_values_respect_v2_contract,true,0
quantity_widened,true,bigint
replay_idempotent,true,16
sales_channel_added,true,per-write mergeSchema (serverless)
sales_channel_values_respect_v2_contract,true,0
unique_business_keys,true,16
v2_supersedes_v1,true,1


transaction_line_id,quantity,loyalty_tier,sales_channel,contract_version
00001c8394e60696d3ae8dd2517c6755ff3fadfb46b12276ef4574ae66efafa2,2,null,null,v1
000040ca23d38cc54100c36a99f626d451a1740a6af4c0b0b0dca5a1399b45d2,2,null,null,v1
00005b3c325c78c5d1d95b540a57613a6d4c1d61d67813f19059db47013880dc,1,null,null,v1
00008cceb3ba0f98407fdaf32e2714a9dea044bdf03489e2285e428854c310c6,5,null,null,v1
0000d3d554c456640a1a557899acb1e48d98528324ae82d2c069e4c7464c70c0,8,STANDARD,null,v2
00023bd5c731e5917d7dde4feb2a2373a97d0c79d9ac02f673f7328feab27337,24,STANDARD,null,v2
00023f001931023af646ac9eed3bd00b6e8cc83b21586471463f86918f6827a2,4,STANDARD,null,v2
0002699038773254fe11aaa8597d218b3298408a2eb44315232a613f0e757be6,2,STANDARD,null,v2
000277eaa7631db9491f906c11fff383a44a19308207592f66fd192c75fb39a8,12,SILVER,ONLINE,v2
00027fa3ffe0bfd366f0e6c6c82af07b88444879f308917bd7e23bcf4f757856,2,SILVER,ONLINE,v2


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-10T16:58:42.000Z,75952854126293,parvinbadalov@yahoo.com,MERGE,"Map(predicate -> [""(transaction_line_id#25105 = transaction_line_id#24924)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1544866556808021),29f3e3de-3c8a-4f39-864b-f8866b3e2be0,0810-153151-u7doqcxo-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2100, materializeSourceTimeMs -> 8, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2062)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-10T16:58:36.000Z,75952854126293,parvinbadalov@yahoo.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(1544866556808021),d9671371-f69d-462c-8a85-ebb03828dc1a,0810-153151-u7doqcxo-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 4153)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-10T16:58:32.000Z,75952854126293,parvinbadalov@yahoo.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(1544866556808021),429e0063-6e94-4f75-a41b-6733a4fd0728,0810-153151-u7doqcxo-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 4138)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-10T16:58:26.000Z,75952854126293,parvinbadalov@yahoo.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(1544866556808021),18a6ff30-7dd5-4449-99cb-249e0640226a,0810-153151-u7doqcxo-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 3908)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-10T16:58:23.000Z,75952854126293,parvinbadalov@yahoo.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1544866556808021),12e23501-18a1-41d4-8f80-41ccb9d2f437,0810-153151-u7doqcxo-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 3630)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-10T16:58:21.000Z,75952854126293,parvinbadalov@yahoo.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""lab.evolution.policy"":""approved_changes_only"",""delta.enableDeletionVectors"":""true"",""delta.enableTypeWidening"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""lab.contract.version"":""v1"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-3e38d410-50e5-48e3-bd9b-a681a802b85d"",""lab.contract.source"":""contracts/online_retail_v1.yml"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-57d06758-f325-434d-8496-9519dddd7276""}, statsOnLoad -> false)",null,List(1544866556808021),686b7de3-bba2-4d76-9224-8cac508d044a,0810-153151-u7doqcxo-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


✅ Controlled schema evolution validation passed using repository contracts v1 → v2.


## What this notebook proved

| Mechanism | Scope | Demonstrated change | Main trade-off |
|---|---|---|---|
| `mergeSchema` | One write | Added `loyalty_tier`; widened `quantity` | Explicit and reviewable |
| `mergeSchema` | One write | Added `sales_channel` | Serverless-compatible and limited to one reviewed write |
| Type widening | Table feature | `INT` to `BIGINT` | Safe widening only; do not narrow silently |
| Contract v2 | Governance boundary | Approved fields and types | Requires review before activation |
| Key-based `MERGE` | Replay behavior | Duplicate retry inserted zero rows | Requires a stable business key |

Session-wide `autoMerge` is discussed but not enabled because it is unavailable on this serverless environment and grants broader evolution permission than this demonstration needs. New columns are not automatically trustworthy merely because Delta can store them. The contract, quality rules, and validation results make the evolution controlled.

## Next notebook

Continue with **`lab04_10_column_mapping.ipynb`**. It will enable Delta column mapping and demonstrate safe column rename and drop operations without rewriting the underlying Parquet data.